# A Bayesian Formulation of Wald’s Sequential Problem

Consider a setting where nature

1. chooses between two distributions $f_0$ and $f_1$
2. and then generates a sequence of IID draws $(z_k)_{k \geq 0}$ from the chosen distribution.

A decision maker wants to know which of the two probability distributions governs these draws.

Observing more draws makes it easier to determine the underlying distribution.

At the same time, there is a cost to observing draws --- each observation requires a fixed cost $c$.

The decision maker pays this cost to observe draws until he feels he is ready to select one of the distributions as the source of random draws.

## A Bayesian formulation.

The decision maker begins  with a prior probability $\pi_{-1}$, representing the probability weight
they put on $f_1$ being the underlying distribution.

We can write this as

$$
\pi_{-1} =
    \mathbb P \{ f = f_1 \mid \textrm{ no observations} \} \in (0, 1)
$$

After observing $ k+1 $ observations $ z_k, z_{k-1}, \ldots, z_0 $, he updates his personal probability that the observations are described by distribution $ f_1 $  to

$$
\pi_k = \mathbb P \{ f = f_1 \mid z_k, z_{k-1}, \ldots, z_0 \}
$$

The sequence $(\pi_k)$ is calculated recursively by applying Bayes’ law:

$$
\pi_{k+1} = \frac{ \pi_k f_1(z_{k+1})}{ (1-\pi_k) f_0(z_{k+1}) + \pi_k f_1 (z_{k+1}) },
\quad k = -1, 0, 1, \ldots
$$

Informally (since we are dealing with densities), you can think of this as 

$$
\mathbb P\{f = f_1 \mid z_{k+1} \}
= \frac{ 
   \mathbb P\{z_{k+1} \mid f = f_1\} \mathbb P\{f = f_1\}}
   {   
       \mathbb P\{z_{k+1} \mid f = f_0\}\mathbb P\{f = f_0\} + 
       \mathbb P\{z_{k+1} \mid f = f_1\}\mathbb P\{f = f_1\}
    }
$$

After observing $ z_k, z_{k-1}, \ldots, z_0 $, the decision-maker's best guess of the distribution of
 $ z_{k+1} $ is

$$
f_{{\pi}_k} (v) = (1-\pi_k) f_0(v) + \pi_k f_1 (v) ,
$$

which  is a mixture of distributions $ f_0 $ and $ f_1 $, with the weight
on $ f_1 $ being the posterior probability that $ f = f_1 $.

We use this expression below when considering whether to observe another draw.

We will solve the decision problem using dynamic programming.

We need the following imports.

In [ ]:
import matplotlib.pyplot as plt
from typing import NamedTuple
import jax
import jax.numpy as jnp
from jax.scipy.special import gamma
from time import time

### Losses and Costs

After observing $ z_k, z_{k-1}, \ldots, z_0 $, the decision-maker
chooses among three distinct actions:

- He decides that $ f = f_0 $ and draws no more $ z $’s  
- He decides that $ f = f_1 $ and draws no more $ z $’s  
- He postpones deciding now and instead chooses to draw a
  $ z_{k+1} $  


Associated with these three actions, the decision-maker can suffer three
kinds of losses:

- A loss $ L_0 $ if he decides $ f = f_0 $ when actually
  $ f=f_1 $  
- A loss $ L_1 $ if he decides $ f = f_1 $ when actually
  $ f=f_0 $  
- A cost $ c $ if he postpones deciding and chooses instead to draw
  another $ z $

### Intuition

Before proceeding,  let’s try to guess what an optimal decision rule might look like.

Suppose at some given point in time that $ \pi $ is close to 1.

Then our prior beliefs and the evidence so far point strongly to $ f = f_1 $.

If, on the other hand, $ \pi $ is close to 0, then $ f = f_0 $ is strongly favored.

Finally, if $ \pi $ is in the middle of the interval $ [0, 1] $, then we are confronted with more uncertainty.

This reasoning suggests a sequential decision rule where we 

- stop and accept $f_0$ if $\pi$ falls below some threshold $B$
- stop and accept $f_1$ if $\pi$ rises above some threshold $A$
- otherwise continue, drawing another observation and reassessing

### A Bellman Equation

Let $ J(\pi) $ be the total loss for a decision-maker with current belief $ \pi $ who chooses optimally.

With some thought, you will agree that $ J $ should satisfy the Bellman equation


<a id='equation-new1'></a>
$$
J(\pi) =
    \min
    \left\{
        \underbrace{\pi L_0}_{ \text{accept } f_0 } \; , \; \underbrace{(1-\pi) L_1}_{ \text{accept } f_1 } \; , \;
        \underbrace{c + \mathbb E  J (\pi') }_{ \text{draw again} }
    \right\} \tag{24.1}
$$

where $ \pi' $ is the random variable defined by Bayes’ Law

$$
\pi' = \kappa(z', \pi) = \frac{ \pi f_1(z')}{ (1-\pi) f_0(z') + \pi f_1 (z') }
$$

when $ \pi $ is fixed and $ z' $ is drawn from the current best guess, which is the distribution $ f $ defined by

$$
    f_{\pi}(v) = (1-\pi) f_0(v) + \pi f_1 (v)
$$

In the Bellman equation, minimization is over three actions:

1. Accept the hypothesis that $ f = f_0 $  
1. Accept the hypothesis that $ f = f_1 $  
1. Postpone deciding and draw again  


We can represent the  Bellman equation as


<a id='equation-optdec'></a>
$$
    J(\pi) 
    =
    \min \left\{ \pi L_0, \; (1-\pi) L_1, \; h(\pi) \right\} \tag{24.2}
$$

where $ \pi \in [0,1] $ and

- $ \pi L_0 $ is the expected loss associated with accepting
  $ f_0 $ 
- $ (1-\pi) L_1 $ is the expected loss associated with accepting
  $ f_1 $
- $ h(\pi) :=  c + \mathbb E [J(\pi')] $; this is the continuation value; i.e.,
  the expected cost associated with drawing one more $ z $.

The optimal decision rule is characterized by two numbers $ A, B \in (0,1)$, where

$$
    \pi L_0 \leq \min \{ (1-\pi) L_1, c + \mathbb E [J(\pi')] \}  
    \quad \iff \quad
    \pi \leq B
$$

and

$$
    (1- \pi) L_1 \leq \min \{ \pi L_0,  c + \mathbb E [J(\pi')] \} 
    \quad \iff \quad
    \pi \geq A
$$

The optimal decision rule is 

$$
\begin{aligned}
\textrm { accept } f=f_1 \textrm{ if } \pi \geq A \\
\textrm { accept } f=f_0 \textrm{ if } \pi \leq B \\
\textrm { draw another }  z \textrm{ if }  B < \pi < A
\end{aligned}
$$

Our aim is to compute the cost function $ J $ as well as  the associated cutoffs $ A $
and $ B $.

It turns out to be convenient to rewrite the Bellman equation in terms of the continuation value function $h$.

To do this we use the definition and the previous Bellman equation to get

<a id='equation-optdec2'></a>
$$
\begin{aligned}
h(\pi) &= c + \mathbb E [J(\pi')] \\
&= c + \mathbb E_{\pi'} \min \{ \pi' L_0, (1 - \pi') L_1, h(\pi') \} \\
&= c + \int \min \{ \kappa(z', \pi) L_0, (1 - \kappa(z', \pi) ) L_1, h(\kappa(z', \pi) ) \} f_\pi (z') dz'
\end{aligned} \tag{24.3}
$$

The equality


<a id='equation-funceq'></a>
$$
h(\pi) =
c + \int \min \{ \kappa(z', \pi) L_0, (1 - \kappa(z', \pi) ) L_1, h(\kappa(z', \pi) ) \} f_\pi (z') dz' \tag{24.4}
$$

is an equation  in an unknown function  $ h $.

Using the functional equation, [(24.4)](#equation-funceq), for the continuation cost, we can back out
optimal choices using the right side of [(24.2)](#equation-optdec).

The functional equation can be solved by iterating with the associated fixed point operator, which is

$$
    Q h(\pi) =
    c + \int 
    \min \{ \kappa(z', \pi) L_0, (1 - \kappa(z', \pi) ) L_1, h(\kappa(z', \pi) ) \} f_\pi (z') dz'
$$

## Implementation

We suppose that $f_0$ and $f_1$ are Beta distributed.

For this reason we introduce the Beta density

In [ ]:
@jax.jit
def p(x, a, b):
    " Beta(a, b) density. "
    r = gamma(a + b) / (gamma(a) * gamma(b))
    return r * x**(a-1) * (1 - x)**(b-1)

Next we make a `NamedTuple` to store the data for a given model.

In [ ]:
class Model(NamedTuple):
    a0: float            # Parameters of f_0 beta distributions
    b0: float            # Parameters of f_0 beta distributions
    a1: float            # Parameters of f_1 beta distributions
    b1: float            # Parameters of f_1 beta distributions
    c:  float            # Cost of another draw
    L0: float            # Cost of selecting f0 when f1 is true
    L1: float            # Cost of selecting f1 when f0 is true
    π_grid: jnp.ndarray  # Grid over π space
    z_grid: jnp.ndarray  # Grid over z space 

Here's a function to create a default instance.

In [ ]:
def create_model_instance(
        c=1.25, a0=1, b0=1, a1=3, b1=1.2, L0=25, L1=25, 
        pi_grid_size=250, z_grid_size=250, seed=1234
    ):
    """
    Create an instance of the model.  The two distributions are assumed to be
    Beta and hence supported on (0, 1).  This is reflected in the fact that
    z_grid is over the space space.

    """
    key = jax.random.PRNGKey(seed)
    ϵ = 1e-6  # Endpoints of open intervals, for stability
    π_grid = jnp.linspace(ϵ, 1 - ϵ, pi_grid_size)
    z_grid = jnp.linspace(ϵ, 1 - ϵ, z_grid_size)
    return Model(
        a0=a0, b0=b0, a1=a1, b1=b1, c=c, L0=L0, L1=L1, 
        π_grid=π_grid, z_grid=z_grid
    )

It will be convenient to define $f_0$ and $f_1$ as densities, given the model

We also define the Bayesian update rule for $\pi$.

In [ ]:
def f0(model, x):
    return p(x, model.a0, model.b0)

def f1(model, x):
    return p(x, model.a1, model.b1)

def κ(model, z, π):
    """
    Updates π using Bayes' rule and the current observation z

    """
    π_f0, π_f1 = (1 - π) * f0(model, z), π * f1(model, z)
    π_new = π_f1 / (π_f0 + π_f1)
    return π_new

We approximate $h$ at a finite grid of possible values of $ \pi $.  

When we evaluate $h$ between grid points, we use linear interpolation.

In [ ]:
@jax.jit
def Q(model, h):
    " Evaluate Qh on the grid of π values in π_grid. "
    L0, L1 = model.L0, model.L1
    c, π_grid, z_grid = model.c, model.π_grid, model.z_grid
    n = len(z_grid)
    hf = lambda π: jnp.interp(π, π_grid, h)
    f_pi = lambda π, z: (1 - π) * f0(model, z) + π * f1(model, z)

    def compute_integral(π):
        # Evaluate the integrand at every z in z_grid
        next_π = κ(model, z_grid, π)
        expected_loss_f0 = next_π * L0        # expected cost of choosing f0
        expected_loss_f1 = (1 - next_π) * L1  # expected cost of choosing f1
        m = jnp.minimum(expected_loss_f0, expected_loss_f1)
        y = jnp.minimum(m, hf(next_π)) * f_pi(π, z_grid)
        # Approximate the integral using a trapezoidal rule
        dz = 1 / (n - 1)
        return dz * (jnp.sum(y) - 0.5 * (y[0] + y[-1]))

    # Compute integral at all π in π_grid, add c, return
    h_new = c + jax.vmap(compute_integral)(π_grid)
    return h_new

To solve the key functional equation, we will iterate using `Q` to find the fixed point

In [ ]:
@jax.jit
def solve_model(model, tol=1e-8, max_iter=1_000):
    " Compute the continuation cost function. "

    def update(state):
        i, error, h = state
        h_new = Q(model, h)
        new_error = jnp.max(jnp.abs(h - h_new))
        i += 1
        return i, new_error, h_new

    def test(state):
        i, error, h = state
        return (i < max_iter) & (error > tol)

    h = jnp.zeros_like(model.π_grid)
    i = 0
    error = tol + 1
    initial_state = i, error, h
    final_state = jax.lax.while_loop(test, update, initial_state)
    i, error, h = final_state
    return i, h

We will also set up a function to compute the cutoffs $ A $ and $ B $
and plot these on our cost function plot

In [ ]:
def find_cutoff_rule(model, h):

    """
    This function takes a continuation cost function and returns the
    corresponding cutoffs of where you transition between continuing and
    choosing a specific model
    """

    π_grid = model.π_grid
    L0, L1 = model.L0, model.L1
    # Evaluate cost at all points on grid for choosing a model
    cost_f0 = π_grid * L0
    cost_f1 = (1 - π_grid) * L1
    
    # Find B: largest π where cost_f0 <= min(cost_f1, h)
    optimal_cost = jnp.minimum(jnp.minimum(cost_f0, cost_f1), h)
    choose_f0 = (cost_f0 <= cost_f1) & (cost_f0 <= h)
    
    if jnp.any(choose_f0):
        B = π_grid[choose_f0][-1]  # Last point where we choose f0
    else:
        assert False, "No point where we choose f0"
    
    # Find A: smallest π where cost_f1 <= min(cost_f0, h)  
    choose_f1 = (cost_f1 <= cost_f0) & (cost_f1 <= h)
    
    if jnp.any(choose_f1):
        A = π_grid[choose_f1][0]  # First point where we choose f1
    else:
        assert False, "No point where we choose f1"

    return (B, A)

## Analysis

Let’s inspect outcomes.

We will be using the default parameterization with distributions like so

In [ ]:
model = create_model_instance()
a0, b0, a1, b1, c, L0, L1, π_grid, z_grid = model

def plot_densities():
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(π_grid, f0(model, π_grid), label="$f_0$")
    ax.plot(π_grid, f1(model, π_grid), label="$f_1$")
    ax.set(ylabel="probability of $z_k$", xlabel="$z_k$", title="Distributions")
    ax.legend()
    plt.show()

Let’s solve the model

In [ ]:
num_iter, h_star = solve_model(model)    # Solve the model
print(f"Model solved in {num_iter} iterations.")

# Timing
start_time = time()
num_iter, h_star = solve_model(model)
h_star.block_until_ready()
elapsed = time() - start_time
print(f"Executed in {elapsed:.3f} seconds")

cost_L0 = π_grid * L0
cost_L1 = (1 - π_grid) * L1
B, A = find_cutoff_rule(model, h_star)

fig, ax = plt.subplots()
ax.plot(π_grid, h_star, label='sample again')
ax.plot(π_grid, cost_L1, label='choose f1')
ax.plot(π_grid, cost_L0, label='choose f0')
ax.plot(π_grid,
       jnp.amin(jnp.column_stack([h_star, cost_L0, cost_L1]), axis=1),
        lw=10, alpha=0.1, color='b', label=r'$J$')
ax.annotate(r"$B$", xy=(B + 0.01, 0.5), fontsize=14)
ax.annotate(r"$A$", xy=(A + 0.01, 0.5), fontsize=14)
plt.vlines(B, 0, (1 - B) * model.L1, linestyle="--")
plt.vlines(A, 0, A * model.L0, linestyle="--")
ax.set(
    xlim=(0, 1), ylim=(0, 0.5 * max(model.L0, model.L1)), 
    ylabel="cost", xlabel=r"$\pi$", 
    title=r"Cost function $J$"
)

plt.legend(frameon=False, loc='lower center')
plt.show()

The cost function $ J $ equals $ \pi L_0 $ for $ \pi \leq B $, and $ (1-\pi) L_1 $ for $ \pi
\geq A $.

The slopes of the two linear pieces of the cost   function $ J(\pi) $ are determined by $ L_0 $
and $ -L_1 $.

The cost function $ J $ is smooth in the interior region, where the posterior
probability assigned to $ f_1 $ is in the indecisive region $ \pi \in (B, A) $.

The decision-maker continues to sample until the probability that he attaches to
model $ f_1 $ falls below $ B $ or above $ A $.